# Bus RavKav — AM-Peak Trips by TAZ

Processes the four RavKav bus-trip files (`Input/BusRavKav/`, Tuesdays 2022-05-03/17/24/31):

1. **Stop → TAZ tagging**: every unique bus stop (by `stop_code`, WGS84 lat/lon) is spatially joined to the `TAZ_North` polygons (781 zones, reprojected to the shapefile's Israeli TM CRS).
2. **Deduplication**: the raw files carry duplicate records (6.9% of rows on 2022-05-03), including exact repeats of the same leg. A **journey** is one `passanger_trip_id` (its first leg by `bus_trip_id` carries the journey `orig`/`dest`); a **leg** is one (`passanger_trip_id`, `bus_trip_id`) pair.
3. **Filters**: `weekday = 3`; hour from the **`bus_trip_hour`** column ∈ {6, 7, 8} (the `date` column carries no time). For the journey OD, the first leg's `bus_trip_hour` (the journey's start hour) decides.
4. **Aggregation**, weighted by `total_boardings` (constant within a journey), averaged over the four days: an OD matrix by TAZ from journey `orig` → `dest` stops, and per-TAZ boardings/alightings from the physical `board`/`alight` stops of each leg.

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os

FILES = ['Input/BusRavKav/trips_table_2022-05-03.csv',
         'Input/BusRavKav/trips_table_2022-05-17.csv',
         'Input/BusRavKav/trips_table_2022-05-24.csv',
         'Input/BusRavKav/trips_table_2022-05-31.csv']
USECOLS = ['passanger_trip_id', 'bus_trip_id', 'weekday', 'bus_trip_hour', 'total_boardings',
           'orig_stop_code', 'orig_stop_lat', 'orig_stop_lon',
           'dest_stop_code', 'dest_stop_lat', 'dest_stop_lon',
           'board_stop_code', 'board_stop_lat', 'board_stop_lon',
           'alight_stop_code', 'alight_stop_lat', 'alight_stop_lon']

taz = gpd.read_file('Input/TAZ_North/TAZ_North.shp')[['TAZ_NUMBER', 'geometry']]
print(f"TAZ polygons: {len(taz)}")

TAZ polygons: 781


## 1. Collect unique stops and tag them by TAZ polygon

In [2]:
stops = {}
frames = []
for f in FILES:
    df = pd.read_csv(f, usecols=USECOLS)
    frames.append(df)
    for role in ['orig', 'dest', 'board', 'alight']:
        s = df[[f'{role}_stop_code', f'{role}_stop_lat', f'{role}_stop_lon']].dropna()
        s.columns = ['stop_code', 'lat', 'lon']
        for code_, lat, lon in s.drop_duplicates('stop_code').itertuples(index=False):
            stops.setdefault(code_, (lat, lon))
    print(f"{f.split('/')[-1]}: {len(df):,} records")

stops_df = pd.DataFrame([(k, v[0], v[1]) for k, v in stops.items()], columns=['stop_code', 'lat', 'lon'])
pts = gpd.GeoDataFrame(stops_df, geometry=gpd.points_from_xy(stops_df['lon'], stops_df['lat']), crs='EPSG:4326').to_crs(taz.crs)
tagged = gpd.sjoin(pts, taz, how='left', predicate='within')
tagged = tagged.drop_duplicates('stop_code')
stop_to_taz = tagged.set_index('stop_code')['TAZ_NUMBER']
os.makedirs('Output/bus', exist_ok=True)
tagged[['stop_code', 'lat', 'lon', 'TAZ_NUMBER']].to_csv('Output/bus/bus_stops_taz.csv', index=False, float_format='%.6f')
print(f"unique stops: {len(stops_df):,} | inside a TAZ_North polygon: {stop_to_taz.notna().sum():,} "
      f"({stop_to_taz.notna().mean():.1%})")

trips_table_2022-05-03.csv: 2,476,772 records


trips_table_2022-05-17.csv: 2,813,312 records


trips_table_2022-05-24.csv: 2,826,013 records


trips_table_2022-05-31.csv: 2,870,481 records


unique stops: 27,186 | inside a TAZ_North polygon: 9,924 (36.5%)


## 2. Deduplicate, filter (weekday 3, bus_trip_hour 6–8), aggregate per day

In [3]:
od_days, board_days, alight_days = [], [], []
for f, df in zip(FILES, frames):
    df = df[df['weekday'] == 3]

    # legs: one row per (journey, bus trip); exact duplicate records dropped
    legs = df.drop_duplicates(['passanger_trip_id', 'bus_trip_id'])
    legs_am = legs[legs['bus_trip_hour'].isin([6, 7, 8])].copy()
    legs_am['b_taz'] = legs_am['board_stop_code'].map(stop_to_taz)
    legs_am['a_taz'] = legs_am['alight_stop_code'].map(stop_to_taz)
    board_days.append(legs_am.dropna(subset=['b_taz']).groupby('b_taz')['total_boardings'].sum())
    alight_days.append(legs_am.dropna(subset=['a_taz']).groupby('a_taz')['total_boardings'].sum())

    # journeys: first leg per passanger_trip_id carries orig/dest and the start hour
    journeys = df.sort_values(['passanger_trip_id', 'bus_trip_id']).drop_duplicates('passanger_trip_id')
    j_am = journeys[journeys['bus_trip_hour'].isin([6, 7, 8])].copy()
    j_am['o_taz'] = j_am['orig_stop_code'].map(stop_to_taz)
    j_am['d_taz'] = j_am['dest_stop_code'].map(stop_to_taz)
    od = j_am.dropna(subset=['o_taz', 'd_taz']).groupby(['o_taz', 'd_taz'])['total_boardings'].sum()
    od_days.append(od)
    print(f"{f.split('/')[-1]}: journeys 6–9 = {j_am['total_boardings'].sum():,.0f} passengers | "
          f"orig+dest in TAZ_North: {od.sum():,.0f} ({od.sum() / j_am['total_boardings'].sum():.1%}) | "
          f"AM boarding events in area: {board_days[-1].sum():,.0f}")

trips_table_2022-05-03.csv: journeys 6–9 = 759,171 passengers | orig+dest in TAZ_North: 132,916 (17.5%) | AM boarding events in area: 145,120


trips_table_2022-05-17.csv: journeys 6–9 = 887,601 passengers | orig+dest in TAZ_North: 148,135 (16.7%) | AM boarding events in area: 162,482


trips_table_2022-05-24.csv: journeys 6–9 = 861,921 passengers | orig+dest in TAZ_North: 145,761 (16.9%) | AM boarding events in area: 160,206


trips_table_2022-05-31.csv: journeys 6–9 = 865,092 passengers | orig+dest in TAZ_North: 146,797 (17.0%) | AM boarding events in area: 161,247


## 3. Average the four days

In [4]:
od_avg = pd.concat(od_days, axis=1).fillna(0).mean(axis=1)
od_matrix = od_avg.unstack().fillna(0)
od_matrix.index = od_matrix.index.astype(int)
od_matrix.columns = od_matrix.columns.astype(int)
od_matrix.index.name = 'orig_taz'
od_matrix.to_csv('Output/bus/bus_od_taz_avg.csv', float_format='%.6g')

ba = pd.concat([pd.concat(board_days, axis=1).fillna(0).mean(axis=1).rename('avg_boardings'),
                pd.concat(alight_days, axis=1).fillna(0).mean(axis=1).rename('avg_alightings')], axis=1).fillna(0)
ba.index = ba.index.astype(int)
ba.index.name = 'TAZ_NUMBER'
ba.to_csv('Output/bus/bus_boardings_alightings_taz.csv', float_format='%.6g')

print(f"OD matrix (avg Tuesday, 6:00–9:00): {od_matrix.shape}, {od_matrix.values.sum():,.0f} passengers/day")
print(f"boardings/alightings: {len(ba)} TAZs | avg daily boardings {ba['avg_boardings'].sum():,.0f}, "
      f"alightings {ba['avg_alightings'].sum():,.0f}")
print("\ntop 10 TAZs by average AM-peak boardings:")
print(ba.sort_values('avg_boardings', ascending=False).head(10).round(1).to_string())

OD matrix (avg Tuesday, 6:00–9:00): (722, 711), 143,402 passengers/day
boardings/alightings: 730 TAZs | avg daily boardings 157,264, alightings 147,632

top 10 TAZs by average AM-peak boardings:
            avg_boardings  avg_alightings
TAZ_NUMBER                               
1219               6478.5          8916.8
1517               3703.0          6527.0
4010               2838.2          1478.8
3907               2675.5           883.8
106                2190.2          1145.5
1408               1919.2          2234.0
1518               1845.5           950.5
4015               1813.2          2257.2
1311               1789.5          2108.5
3615               1773.0          1282.2


## Notes

- The hour filter uses the **`bus_trip_hour`** column (per the data owner's guidance); the `date` column carries no time. For journey-level OD, the journey's first leg decides its hour.
- The OD matrix counts each journey once (deduplicated by `passanger_trip_id`, orig→dest stops, both ends inside `TAZ_North`). The boardings/alightings table counts each deduplicated leg's physical board/alight events, so a transfer journey contributes at each boarding point.
- All aggregates are weighted by `total_boardings` (passengers per record; constant within a journey's legs) and averaged over the four Tuesdays.
- A fifth date file (`Input/trips_table_2022-05-10.csv`) sits outside the `BusRavKav` directory and is **not** included, per the four-file instruction.